# Instalación de Aletheia

**Instalación de dependencias**

In [ ]:
# Instalar dependencias
!pip install imageio numpy scipy tensorflow scikit-learn pandas hdf5storage h5py matplotlib steganogan python-magic efficientnet Pillow

In [ ]:
!apt update
!apt install outguess steghide liboctave-dev imagemagick
!apt install octave-image octave-signal octave-nan

In [ ]:
# Clonar sin descargar los blobs
#!git clone --filter=blob:none https://github.com/daniellerch/aletheia.git
#
# En lugar de clone usaremos una descarga wget que es más rápida, pero no instala toda la suite.
# Si queréis usar la suite completa, debéis ejecutar el clone que está comentado.

!wget -O- https://github.com/daniellerch/uoc/releases/download/v1/aletheia.tar.gz | tar -xz
%cd aletheia

**Test de Aletheia**

Realizamos un ataque SPA sobre una de las imágenes de muetra de Aletheia:

In [ ]:
!./aletheia.py spa sample_images/lena_lsbr_0.3.png

Usamos un modelo de machine learning para detectar Steghide, sobre imágenes de ejemplo proporcionadas por Aletheia.
Cada línea muestra el nombre de la imágen seguido de la probabilidad de que la imagen contenga información oculta usando Steghide.


In [ ]:
!./aletheia.py effnetb0-predict actors/A2 aletheia-models/effnetb0-A-alaska2-steghide.h5 0 2>/dev/null

# Estegoanálisis Básico

**Descarga de imágenes de prueba**

In [ ]:
!git clone https://github.com/daniellerch/uoc.git


**Instalamos OpenStego**

In [ ]:
!wget -q https://github.com/syvaidya/openstego/releases/download/openstego-0.8.6/openstego-0.8.6.zip
!unzip -q openstego-0.8.6.zip

**Ocultamos un mensaje usando OpenStego**

Usamos un payload aproximado del 5% del número de píxeles, a un bit por píxel.

In [ ]:
!head -c 4916 /dev/urandom > secret.bin # Payload 5% -> 512*512*3*0.05/8 = 4916 bits
!java -jar openstego-0.8.6/lib/openstego.jar embed \
    -mf secret.bin \
    -cf uoc/072026/cover-png/06017.png \
    -sf uoc/072026/06017_openstego.png

**Detectamos la presencia del mensaje mediante un ataque SPA**

In [ ]:
!./aletheia.py spa uoc/072026/06017_openstego.png 0.03

**Ocultamos un mensaje usando StegHide**

Usamos un payload aproximado del 5% de la capacidad de Steghide, a un bit por píxel.


In [ ]:
# !steghide info uoc/072026/cover-jpg/02452.jpg # Capacity 3.9KB
!head -c 200 /dev/urandom > secret.bin # Payload 5% -> 3.9*1024*0.05 = 200 bytes
!steghide embed -cf uoc/072026/cover-jpg/02452.jpg -ef secret.bin -sf uoc/072026/02452_steghide.jpg -p ""

In [ ]:
!./aletheia.py effnetb0-predict uoc/072026/02452_steghide.jpg aletheia-models/effnetb0-A-alaska2-steghide.h5 0 2>/dev/null

**Instalamos HStego**

In [ ]:
!pip install imageio numpy scipy pycryptodome numba Pillow
!pip install git+https://github.com/daniellerch/hstego.git@v0.6

**Ocultamos un mensaje usando Hstego en imágenes PNG (UNIWARD)**

Usamos un payload aproximado del 5% del número de píxeles, a un bit por píxel.

In [ ]:
!hstego.py capacity uoc/072026/cover-png/18510.png # 4700 - 80 (header)
!head -c 4620 /dev/urandom > secret.bin # Max HStego payload ~ 5%
!hstego.py embed secret.bin uoc/072026/cover-png/18510.png uoc/072026/18510_hstego_uniward.png myp4ssw0rd

**Intentamos detectar la presencia del mensaje usando un modelo ML**

In [ ]:
!./aletheia.py effnetb0-predict uoc/072026/18510_hstego_uniward.png aletheia-models/effnetb0-A-alaska2-uniw.h5 0 2>/dev/null

**Ocultamos un mensaje usando HStego en imágenes JPG (J-UNIWARD+Cost polarization)**

Usamos un payload aproximado del 5% del número de coeficientes DCT diferentes de cero, a un bit por coeficiente.

In [ ]:

!hstego.py capacity uoc/072026/cover-jpg/08020.jpg # 844 bytes - 80 (header)
!head -c 764 /dev/urandom > secret.bin # Max HStego payload ~ 5%
!hstego.py embed secret.bin uoc/072026/cover-jpg/08020.jpg uoc/072026/08020_hstego_juniw+cp.jpg myp4ssw0rd

**Intentamos detectar la presencia del mensaje usando un modelo ML**

In [ ]:
!./aletheia.py effnetb0-predict uoc/072026/08020_hstego_juniw+cp.jpg aletheia-models/effnetb0-A-alaska2-juniw+wiener.h5 0 2>/dev/null

# Análisis de una herramienta desconocida

**Estudio de las modificaciones realizadas por OpenStego**

Para saber qué método usar para analizar una herramienta, necesitamos conocer la forma en la que realiza la inserción. Si la herramienta no documenta cómo realiza estas modificaciones, podemos estudiar los cambios. En el siguiente ejemplo, vemos el tipo de modificaciones que realiza OpenStego. Para ello, imprimimos las diferencias entre los píxeles de la imagen cover y los píxeles de la imagen stego.

Las modificaciones en píxeles impares siempre restan 1, las modificaciones en píxeles parese, siempre suman 1. Esto nos indica que se está usando LSB replacement. Por lo tanto, se puede detectar usando ataques estructurados como SPA, RS o Wighted Stego.

In [ ]:
!./aletheia.py print-diffs uoc/072026/cover-png/06017.png uoc/072026/06017_openstego.png 2>/dev/null | head -20

**Estudio de las modificaciones realizadas por HStego**

Para no ser vulnerable a los ataques estructurados es necesario realizar modificaciónes sumando y restando 1 (aleatoriamente) en lugar de susutitur el LSB, como hace OpenStego. Esta técnicas se conoce como LSB Matching y es la base de métodos más avanzados, como los usados por HStego.

Cuando esto ocurre, suele ser necesario entrenar un model ML a medida.


In [ ]:
!./aletheia.py print-diffs uoc/072026/cover-png/18510.png uoc/072026/18510_hstego_uniward.png 2>/dev/null | head -20

**Estudio de las modificaciones realizadas por Steghide**

A continuación podemos ver como son alterados los coeficientes de Steghide y como se altera el histograma. Las modificaciones conservan la estadística de la imagen lo suficiente como para que sean necesarios ataques de tipo ML para detectarlo.

In [ ]:
!./aletheia.py print-dct-diffs uoc/072026/cover-jpg/02452.jpg uoc/072026/02452_steghide.jpg 2>/dev/null | head -20

In [ ]:
!./aletheia.py print-dct-diffs uoc/072026/cover-jpg/02452.jpg uoc/072026/02452_steghide.jpg 2>/dev/null #| head -20

**Estudio de las modificaciones realizadas por HStego (JPEGJ-UNIWARD + Cost Polarization)**

A continuación podemos ver como son alterados los coeficientes de HStego y como se altera el histograma. Las modificaciones conservan la estadística de la imagen lo suficiente como para que sean necesarios ataques de tipo ML para detectarlo.

In [ ]:
!./aletheia.py print-dct-diffs uoc/072026/cover-jpg/08020.jpg uoc/072026/08020_hstego_juniw+cp.jpg  2>/dev/null #| head -20

**Esteganografía adaptativa**

Los métodos avanzados de esteganografía se adaptan al contenido de la imagen, ocultando la información en las zonas más difíciles de detectar. Normalmentem en ejes y texturas.

A continuación, vemos la diferencias al incrustar usando OpenStego, Steghide y HStego marcada con puntos rojos. En el caso de JPEG, las direncias suelen afectar a todo el bloque de 8x8 píxeles afectado, aunque también sirve para ver como el algoritmo se adapta a las zonas más texturizadas (HStego)

In [ ]:
import imageio.v3 as iio
import matplotlib.pyplot as plt

def plot_diff(cover_path, stego_path, output_path):

    cover = iio.imread(cover_path)
    stego = iio.imread(stego_path)

    out = cover.mean(axis=2).astype("uint8")
    out = plt.cm.gray(out)[..., :3]
    out[(cover != stego).any(axis=2)] = [1, 0, 0]   # rojo

    iio.imwrite(output_path, (out*255).astype("uint8"))
    plt.figure(figsize=(5,5))
    plt.imshow(out)
    plt.axis("off")
    plt.show()

plot_diff("uoc/072026/cover-png/06017.png", "uoc/072026/06017_openstego.png", "out.png")
plot_diff("uoc/072026/cover-png/18510.png", "uoc/072026/18510_hstego_uniward.png", "out.png")
plot_diff("uoc/072026/cover-jpg/02452.jpg" , "uoc/072026/02452_steghide.jpg", "out.png")
plot_diff("uoc/072026/cover-jpg/08020.jpg", "uoc/072026/08020_hstego_juniw+cp.jpg", "out.png")

